# 3.2 · Divide and conquer: mergesort, quicksort and quickselect

*Sorting* · 16 exercises

Split, solve, combine: mergesort, quicksort and quickselect, and what each costs in the best and worst case.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ch-3.2) · [Answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ch-3.2) · [Concept guide](https://bloomcode.github.io/dsa/guide/#ch-3.2)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.
import random

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    import re as _re

    class _Raises:
        def __init__(self, kind, match=None):
            self.kind = kind
            self.match = match
            self.value = None

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            if not issubclass(kind, self.kind):
                return False
            self.value = value
            if self.match is not None and not _re.search(self.match, str(value)):
                raise AssertionError(f"exception message {str(value)!r} did not match {self.match!r}")
            return True

    class pytest:  # noqa: N801
        raises = _Raises

def _check_cmp(l, r, op, ls, rs):
    __tracebackhide__ = True
    passed = False
    if op == "==": passed = (l == r)
    elif op == "!=": passed = (l != r)
    elif op == "<": passed = (l < r)
    elif op == "<=": passed = (l <= r)
    elif op == ">": passed = (l > r)
    elif op == ">=": passed = (l >= r)
    elif op == "is": passed = (l is r)
    elif op == "is not": passed = (l is not r)
    elif op == "in": passed = (l in r)
    elif op == "not in": passed = (l not in r)
    if not passed:
        import inspect
        caller = inspect.currentframe().f_back
        locs = {k: v for k, v in caller.f_locals.items()
                if not k.startswith("_") and type(v) in (int, float, str, bool, list, tuple, dict, set)}
        lines = [
            f"Assertion Failed: {ls} {op} {rs}",
            f"  Expected: {rs} -> {r!r} ({type(r).__name__})",
            f"  Got:      {ls} -> {l!r} ({type(l).__name__})"
        ]
        if locs:
            lines.append("  Context:  " + ", ".join(f"{k} = {v!r}" for k, v in locs.items()))
        if isinstance(l, (list, tuple, str)) and isinstance(r, (list, tuple, str)):
            if len(l) != len(r):
                lines.append(f"  Length mismatch: got {len(l)} items, expected {len(r)}")
            if isinstance(l, (list, tuple)) and isinstance(r, (list, tuple)):
                for i, (a, b) in enumerate(zip(l, r)):
                    if a != b:
                        lines.append(f"  First diff at index {i}: got {a!r}, expected {b!r}")
                        break
        raise AssertionError(chr(10) + chr(10).join(lines))

def _check_expr(val, expr):
    __tracebackhide__ = True
    if not val:
        raise AssertionError(chr(10) + f"Assertion Failed: expected {expr} to be truthy, got {val!r}")

try:
    _ip = get_ipython()
    if _ip is not None:
        import ast as _ast

        class _AssertRewriter(_ast.NodeTransformer):
            def __init__(self):
                self._tid = 0

            def visit_Assert(self, node):
                test = node.test
                if isinstance(test, _ast.BoolOp) and isinstance(test.op, _ast.And):
                    res = []
                    for sub in test.values:
                        res.extend(self.visit_Assert(_ast.Assert(test=sub, msg=node.msg)))
                    return res
                if isinstance(test, _ast.Compare) and len(test.ops) == 1:
                    op_map = {_ast.Eq: "==", _ast.NotEq: "!=", _ast.Lt: "<", _ast.LtE: "<=",
                              _ast.Gt: ">", _ast.GtE: ">=", _ast.Is: "is", _ast.IsNot: "is not",
                              _ast.In: "in", _ast.NotIn: "not in"}
                    op_cls = type(test.ops[0])
                    if op_cls in op_map:
                        self._tid += 1
                        la, ra = f"_a_{self._tid}", f"_b_{self._tid}"
                        return [
                            _ast.Assign(targets=[_ast.Name(id=la, ctx=_ast.Store())], value=test.left),
                            _ast.Assign(targets=[_ast.Name(id=ra, ctx=_ast.Store())], value=test.comparators[0]),
                            _ast.Expr(value=_ast.Call(
                                func=_ast.Name(id="_check_cmp", ctx=_ast.Load()),
                                args=[_ast.Name(id=la, ctx=_ast.Load()), _ast.Name(id=ra, ctx=_ast.Load()),
                                      _ast.Constant(value=op_map[op_cls]), _ast.Constant(value=_ast.unparse(test.left)),
                                      _ast.Constant(value=_ast.unparse(test.comparators[0]))],
                                keywords=[]
                            ))
                        ]
                return [_ast.Expr(value=_ast.Call(
                    func=_ast.Name(id="_check_expr", ctx=_ast.Load()),
                    args=[test, _ast.Constant(value=_ast.unparse(test))],
                    keywords=[]
                ))]

        if not any(isinstance(t, _AssertRewriter) for t in _ip.ast_transformers):
            _ip.ast_transformers.append(_AssertRewriter())
except NameError:
    pass

### 3.2.1 · Partition  `Core`

**Task.** Write `partition(arr, left, right)` using the rightmost element as pivot: a left pointer moves right past values smaller than the pivot, a right pointer moves left past values larger, swap, repeat; finally swap the pivot into the left pointer's spot. Return the pivot's final index.

**Example.** `[0, 5, 2, 1, 6, 3]` -> pivot 3 ends at index 3, values left of it are all < 3, right of it all > 3

**Big O.** O(N) for one partition.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-3-2-1) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-3-2-1)

In [ ]:
# 3.2.1 Partition
def partition(arr, left, right):
    raise NotImplementedError

In [ ]:
# Test helpers: run once.
def random_lists(seed, n=200):
    rng = random.Random(seed)
    return [[rng.randint(0, 9) for _ in range(rng.randint(0, 25))] for _ in range(n)]


def rand_lists(n=60, seed=0):
    rng = random.Random(seed)
    return [[rng.randint(-30, 30) for _ in range(rng.randint(0, 40))] for _ in range(n)]

In [ ]:
def test_ex_3_2_1():
    a = [0, 5, 2, 1, 6, 3]
    p = partition(a, 0, 5)
    assert p == 3
    assert all((x < 3 for x in a[:p]))
    assert all((x > 3 for x in a[p + 1:]))


test_ex_3_2_1()
print('3.2.1 passed')

### 3.2.2 · Quicksort  `Core`

**Task.** Write `quicksort(arr, left=0, right=None)`: partition, then recurse on the sub-arrays left and right of the pivot. Base case: fewer than two elements.

**Example.** `[0, 5, 2, 1, 6, 3]` -> `[0, 1, 2, 3, 5, 6]`

**Big O.** O(N log N) average, O(N^2) worst case (pivot is always the smallest or largest, e.g. sorted input with a rightmost pivot).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-3-2-2) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-3-2-2)

In [ ]:
# 3.2.2 Quicksort
def quicksort(arr, left=0, right=None):
    raise NotImplementedError

In [ ]:
def test_ex_3_2_2():
    for data in random_lists(1):
        assert quicksort(list(data)) == sorted(data)


test_ex_3_2_2()
print('3.2.2 passed')

### 3.2.3 · Quickselect  `Core`

**Task.** Write `quickselect(arr, k)` returning the k-th smallest value (k = 0 is the minimum) without fully sorting: partition, then recurse into only the side that holds index k.

**Example.** `[0, 50, 20, 10, 60, 30]`, k = 1 -> `10`

**Big O.** O(N) average (N + N/2 + N/4 ... = 2N), O(N^2) worst case.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-3-2-3) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-3-2-3)

In [ ]:
# 3.2.3 Quickselect
def quickselect(arr, k, left=0, right=None):
    raise NotImplementedError

In [ ]:
def test_ex_3_2_3():
    assert quickselect([0, 50, 20, 10, 60, 30], 1) == 10
    rng = random.Random(2)
    for data in random_lists(3):
        if data:
            k = rng.randrange(len(data))
            assert quickselect(list(data), k) == sorted(data)[k]


test_ex_3_2_3()
print('3.2.3 passed')

### 3.2.4 · Closest pair after sorting  `Core`

**Task.** Write `closest_gap_sorted(nums)` for at least two numbers: sort a copy with your quicksort, then compare only neighbours.

**Example.** `[10, 4, 15, 7]` -> `3`

**Big O.** O(N log N). Once sorted, the closest pair is always adjacent, so N - 1 comparisons replace 1.1.14's every pair.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-3-2-4) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-3-2-4)

In [ ]:
# 3.2.4 Closest pair after sorting
def closest_gap_sorted(nums):
    raise NotImplementedError

In [ ]:
def test_ex_3_2_4():
    data = [10, 4, 15, 7]
    assert closest_gap_sorted(data) == 3
    assert data == [10, 4, 15, 7]
    assert closest_gap_sorted([1, 100]) == 99
    assert closest_gap_sorted([5, 5, 9]) == 0
    assert closest_gap_sorted([-3, 8, -1]) == 2


test_ex_3_2_4()
print('3.2.4 passed')

### 3.2.5 · Do any meetings overlap?  `Practice`

**Task.** Write `any_overlap(meetings)` for a list of `[start, end]` pairs (a meeting may start exactly when another ends). Sort by start (the built-in sort is fine), then check each meeting against the one before it.

**Example.** `[[9, 11], [13, 15], [10, 12]]` -> `True`; `[[9, 10], [10, 11]]` -> `False`

**Big O.** O(N log N) for the sort, then one O(N) pass.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-3-2-5) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-3-2-5)

In [ ]:
# 3.2.5 Do any meetings overlap?
def any_overlap(meetings):
    raise NotImplementedError

In [ ]:
def test_ex_3_2_5():
    assert any_overlap([[9, 11], [13, 15], [10, 12]])
    assert not any_overlap([[9, 10], [10, 11]])
    assert not any_overlap([])
    assert not any_overlap([[1, 5]])
    assert any_overlap([[1, 10], [2, 3], [11, 12]])


test_ex_3_2_5()
print('3.2.5 passed')

### 3.2.6 · Count distinct values by sorting  `Core`

**Task.** Write `count_distinct_sorted(nums)`: sort a copy with your quicksort, then count the positions whose value differs from the one before.

**Example.** `[5, 9, 3, 5, 9, 9]` -> `3`

**Big O.** O(N log N) time, with no hash table: after sorting, equal values sit side by side.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-3-2-6) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-3-2-6)

In [ ]:
# 3.2.6 Count distinct values by sorting
def count_distinct_sorted(nums):
    raise NotImplementedError

In [ ]:
def test_ex_3_2_6():
    assert count_distinct_sorted([5, 9, 3, 5, 9, 9]) == 3
    assert count_distinct_sorted([]) == 0
    assert count_distinct_sorted([7, 7, 7]) == 1
    assert count_distinct_sorted([3, -3, 0]) == 3


test_ex_3_2_6()
print('3.2.6 passed')

### 3.2.7 · Median with quickselect  `Practice`

**Task.** Write `median(nums)` using `quickselect`. For an even count, average the two middle values.

**Example.** `[7, 1, 3]` -> `3`; `[4, 1, 3, 2]` -> `2.5`

**Big O.** O(N) average.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-3-2-7) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-3-2-7)

In [ ]:
# 3.2.7 Median with quickselect
def median(nums):
    raise NotImplementedError

In [ ]:
def test_ex_3_2_7():
    assert median([7, 1, 3]) == 3
    assert median([4, 1, 3, 2]) == 2.5


test_ex_3_2_7()
print('3.2.7 passed')

### 3.2.8 · Three-way partition for repeated values  `Stretch`

**Task.** Write `quicksort3(arr)` that sorts in place and returns `arr`, partitioning each range into values less than, equal to and greater than the pivot (the first value of the range), then recursing only on the outer two parts.

**Example.** `[3, 1, 3, 2, 3, 1]` -> `[1, 1, 2, 3, 3, 3]`

**Big O.** O(N log N) expected. With only D distinct values it drops to about O(N * D), because the block equal to the pivot is never touched again.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-3-2-8) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-3-2-8)

In [ ]:
# 3.2.8 Three-way partition for repeated values
def quicksort3(arr, lo=0, hi=None):
    raise NotImplementedError

In [ ]:
def test_ex_3_2_8():
    data = [3, 1, 3, 2, 3, 1]
    assert quicksort3(data) is data
    assert data == [1, 1, 2, 3, 3, 3]
    many = [i * 7919 % 5 for i in range(500)]
    assert quicksort3(list(many)) == sorted(many)
    assert quicksort3([]) == []
    assert quicksort3([4]) == [4]
    for data in random_lists(5):
        assert quicksort3(list(data)) == sorted(data)


test_ex_3_2_8()
print('3.2.8 passed')

### 3.2.9 · Merge two sorted arrays  `Core`

**Task.** Write `merge(a, b)` returning one sorted list from two sorted lists, using two indices that advance through them. No `sorted()`.

**Example.** `merge([1, 4, 9], [2, 3, 10, 11])` -> `[1, 2, 3, 4, 9, 10, 11]`

**Big O.** O(N + M) time, O(N + M) space.

**Note.** Use `<=` when comparing so equal items from `a` come first. That one character is what makes mergesort stable (3.2.15).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-3-2-9) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-3-2-9)

In [ ]:
# 3.2.9 Merge two sorted arrays
def merge(a, b):
    raise NotImplementedError

In [ ]:
def test_ex_3_2_9():
    assert merge([1, 4, 9], [2, 3, 10, 11]) == [1, 2, 3, 4, 9, 10, 11]
    assert merge([], [1]) == [1]
    assert merge([], []) == []


test_ex_3_2_9()
print('3.2.9 passed')

### 3.2.10 · Count the comparisons in a merge  `Core`

**Task.** Write `merge_comparisons(a, b)` returning `(merged, comparisons)`. Find inputs that need the fewest and the most comparisons.

**Example.** `merge_comparisons([1, 2, 3], [4, 5, 6])[1]` -> `3`; `merge_comparisons([1, 3, 5], [2, 4, 6])[1]` -> `5`

**Big O.** Between min(N, M) and N + M - 1 comparisons.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-3-2-10) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-3-2-10)

In [ ]:
# 3.2.10 Count the comparisons in a merge
def merge_comparisons(a, b):
    raise NotImplementedError

In [ ]:
def test_ex_3_2_10():
    assert merge_comparisons([1, 2, 3], [4, 5, 6])[1] == 3
    assert merge_comparisons([1, 3, 5], [2, 4, 6])[1] == 5
    assert merge_comparisons([1, 3, 5], [2, 4, 6])[0] == [1, 2, 3, 4, 5, 6]


test_ex_3_2_10()
print('3.2.10 passed')

### 3.2.11 · Mergesort, top-down  `Core`

**Task.** Write `mergesort(arr)` returning a new sorted list: split in half, sort each half recursively, then `merge` them. Base case: length 0 or 1.

**Example.** `mergesort([38, 27, 43, 3, 9, 82, 10])` -> `[3, 9, 10, 27, 38, 43, 82]`

**Big O.** O(N log N) in every case: log2 N levels of splitting, O(N) merging per level. O(N) extra space.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-3-2-11) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-3-2-11)

In [ ]:
# 3.2.11 Mergesort, top-down
def mergesort(arr):
    raise NotImplementedError

In [ ]:
def test_ex_3_2_11():
    assert mergesort([38, 27, 43, 3, 9, 82, 10]) == [3, 9, 10, 27, 38, 43, 82]
    for data in rand_lists(seed=1):
        assert mergesort(data) == sorted(data)


test_ex_3_2_11()
print('3.2.11 passed')

### 3.2.12 · Mergesort with one shared buffer  `Practice`

**Task.** Write `mergesort_in_place(arr)` that sorts `arr` itself, recursing on index ranges and merging into one auxiliary list allocated once, instead of slicing at every level.

**Example.** `a = [5, 2, 4, 6, 1, 3]; mergesort_in_place(a)` -> `a` is `[1, 2, 3, 4, 5, 6]`

**Big O.** O(N log N) time, O(N) extra space total instead of fresh slices per call.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-3-2-12) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-3-2-12)

In [ ]:
# 3.2.12 Mergesort with one shared buffer
def mergesort_in_place(arr):
    raise NotImplementedError

In [ ]:
def test_ex_3_2_12():
    a = [5, 2, 4, 6, 1, 3]
    mergesort_in_place(a)
    assert a == [1, 2, 3, 4, 5, 6]
    for data in rand_lists(seed=2):
        a = list(data)
        mergesort_in_place(a)
        assert a == sorted(data)


test_ex_3_2_12()
print('3.2.12 passed')

### 3.2.13 · Mergesort, bottom-up  `Core`

**Task.** Write `mergesort_bottom_up(arr)` with no recursion: merge runs of width 1 into width 2, then 4, 8, and so on, until one run covers the list.

**Example.** `mergesort_bottom_up([5, 1, 4, 2, 8, 0, 2])` -> `[0, 1, 2, 2, 4, 5, 8]`

**Big O.** O(N log N) time, O(N) space, and no call stack at all.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-3-2-13) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-3-2-13)

In [ ]:
# 3.2.13 Mergesort, bottom-up
def mergesort_bottom_up(arr):
    raise NotImplementedError

In [ ]:
def test_ex_3_2_13():
    for data in rand_lists(seed=3):
        assert mergesort_bottom_up(data) == sorted(data)


test_ex_3_2_13()
print('3.2.13 passed')

### 3.2.14 · Count inversions while merging  `Practice`

**Task.** An inversion is a pair i < j with arr[i] > arr[j]. Write `count_inversions(arr)` by adding, every time an item from the right half is taken first, the number of items still waiting in the left half.

**Example.** `count_inversions([2, 4, 1, 3, 5])` -> `3`; a reversed list of 5 -> `10`

**Big O.** O(N log N), versus O(N^2) for checking every pair.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-3-2-14) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-3-2-14)

In [ ]:
# 3.2.14 Count inversions while merging
def count_inversions(arr):
    raise NotImplementedError

In [ ]:
def test_ex_3_2_14():
    assert count_inversions([2, 4, 1, 3, 5]) == 3
    assert count_inversions([5, 4, 3, 2, 1]) == 10
    for data in rand_lists(seed=4):
        brute = sum((1 for i in range(len(data)) for j in range(i + 1, len(data)) if data[i] > data[j]))
        assert count_inversions(data) == brute


test_ex_3_2_14()
print('3.2.14 passed')

### 3.2.15 · Stability, with records  `Core`

**Task.** Write `mergesort_by(records, key)` that sorts records by `key(record)` and keeps records with equal keys in their original order. Show that quicksort from Chapter 3.2 does not promise this.

**Example.** `mergesort_by([("ann", 3), ("bo", 1), ("cy", 3), ("di", 1)], key=lambda r: r[1])` -> `[('bo', 1), ('di', 1), ('ann', 3), ('cy', 3)]`

**Big O.** O(N log N).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-3-2-15) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-3-2-15)

In [ ]:
# 3.2.15 Stability, with records
def mergesort_by(records, key):
    raise NotImplementedError

In [ ]:
def test_ex_3_2_15():
    records = [('ann', 3), ('bo', 1), ('cy', 3), ('di', 1)]
    assert mergesort_by(records, key=lambda r: r[1]) == [('bo', 1), ('di', 1), ('ann', 3), ('cy', 3)]


test_ex_3_2_15()
print('3.2.15 passed')

### 3.2.16 · Hybrid: insertion sort for small pieces  `Stretch`

**Task.** Write `hybrid_sort(arr, cutoff=16)`: mergesort that switches to insertion sort when a piece has `cutoff` items or fewer. Benchmark it against plain mergesort in Chapter 1.3.

**Example.** `hybrid_sort([9, 8, 7, 6, 5, 4, 3, 2, 1, 0], cutoff=4)` -> `[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]`

**Big O.** Still O(N log N); insertion sort wins on tiny inputs because its constant factor is small. Python's own Timsort does the same trick.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-3-2-16) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-3-2-16)

In [ ]:
# 3.2.16 Hybrid: insertion sort for small pieces
def hybrid_sort(arr, cutoff=16):
    raise NotImplementedError

In [ ]:
def test_ex_3_2_16():
    assert hybrid_sort([9, 8, 7, 6, 5, 4, 3, 2, 1, 0], cutoff=4) == list(range(10))
    for data in rand_lists(seed=5):
        assert hybrid_sort(data, 4) == sorted(data)


test_ex_3_2_16()
print('3.2.16 passed')